In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model, Sequential

import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from umap import UMAP
from tqdm import tqdm
fnames = glob("clips/*npy")
len(fnames)

In [2]:
N = 800  # length of 

def getbrokenfile(fnames):
    while True:
        f = random.choice(fnames)
        x = np.load(f)
        if(x.shape[0]==1_000_000):
            return x
        else:
            pass


In [3]:
import math
import random

class MyGenerator(Sequence):
    def __init__(self, fnames, batch_size=64):
        self.fnames = fnames
        self.batch_size= 64
        
    def __len__(self):
        return math.ceil(len(self.fnames) / self.batch_size)
    
    def __getitem__(self, idx):
        x_train = []
        y_train = []
        for i in range(self.batch_size):
            a = getbrokenfile(self.fnames)
            a = a-a.min()
            a = a/a.max()
            idx = random.choice(list(range(0, a.shape[0]-N) ))
            aa = a[idx:idx+N]
            b = getbrokenfile(self.fnames)
            b = b-b.min()
            b = b/b.max()
 
            idx = random.choice(list(range(0, b.shape[0]-N )))
            bb = b[idx:idx+N]
            
            choice = random.choice([0,1])
            y_train.append(choice)
            idx = random.choice(list(range(0, a.shape[0]-N )))
             
            if(choice==0):
                #source is A
                clip = a[idx:idx+N]
            else:
                clip = b[idx:idx+N]
            assert len(clip)==N
             
            x_train.append(np.hstack([aa,bb,clip]))
        x_train = np.array(x_train)
        y_train = np.array(y_train)
        return x_train, y_train     
        
mygen = MyGenerator(fnames)


In [7]:
def buildABModel(N):
    # size of model is N
    model = Sequential()
    model.add(Dense(768, input_shape=(N*3,), activation='relu'))
    model.add(Dense(300, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(100, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(.3))
    model.add(Dense(1, activation='sigmoid'))
    return model
   
model = buildABModel(N)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['acc'])
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_10 (Dense)            (None, 768)               1843968   
                                                                 
 dense_11 (Dense)            (None, 300)               230700    
                                                                 
 batch_normalization_2 (Batc  (None, 300)              1200      
 hNormalization)                                                 
                                                                 
 dense_12 (Dense)            (None, 100)               30100     
                                                                 
 dense_13 (Dense)            (None, 50)                5050      
                                                                 
 dropout_2 (Dropout)         (None, 50)                0         
                                                      

In [8]:
history = model.fit(mygen, batch_size=128, epochs=12 )

Epoch 1/12
362/362 [==============================] - 3899s 11s/step - loss: 0.3524 - acc: 0.8654
Epoch 2/12
362/362 [==============================] - 3588s 10s/step - loss: 0.3007 - acc: 0.8772
Epoch 3/12
362/362 [==============================] - 3368s 9s/step - loss: 0.2725 - acc: 0.8820
Epoch 4/12
362/362 [==============================] - 3387s 9s/step - loss: 0.2634 - acc: 0.8842
Epoch 5/12
362/362 [==============================] - 3400s 9s/step - loss: 0.2701 - acc: 0.8835
Epoch 6/12
362/362 [==============================] - 3420s 9s/step - loss: 0.2626 - acc: 0.8859
Epoch 7/12
362/362 [==============================] - 3392s 9s/step - loss: 0.2401 - acc: 0.8912
Epoch 8/12
362/362 [==============================] - 3374s 9s/step - loss: 0.2285 - acc: 0.8970
Epoch 9/12
362/362 [==============================] - 3391s 9s/step - loss: 0.2271 - acc: 0.9034
Epoch 12/12
362/362 [==============================] - 3387s 9s/step - loss: 0.2166 - acc: 0.9053


In [ ]:
inp = Input(shape=(4096,))
x = Dense(1512, activation='relu')(inp)
x = Dense(128, activation='relu')(x)
emb = Dense(20, activation='relu')(x)
x = Dense(128, activation='relu')(emb)
x = Dropout(.2)(x)
x = Dense(1512, activation='relu')(x)
out = Dense(4096, activation='sigmoid')(x)

model = Model(inp, out)
emb_model=Model(inp, emb)
model.compile(loss='mse', optimizer='adam', metrics=['mse'])

In [ ]:
model.fit(x_train, x_train, batch_size=64, epochs=6, validation_split=.1)

In [ ]:
y_hat = emb_model.predict(x_train)

In [ ]:
mapper = UMAP().fit_transform(y_hat)

In [ ]:
plt.scatter(mapper[:,0], mapper[:,1], s=1)